In [1]:
import numpy as np
import yfinance as yf


In [6]:
nifty = yf.download("^NSEI", start="2016-01-01", end="2016-12-31", interval="1d")
print(nifty.head())
nifty.to_csv("nifty50_data2016.csv")

C:\Users\RITANKAR\AppData\Local\Temp\ipykernel_4736\3270839231.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI", start="2016-01-01", end="2016-12-31", interval="1d")
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open  Volume
Ticker            ^NSEI        ^NSEI        ^NSEI        ^NSEI   ^NSEI
Date                                                                  
2016-01-04  7791.299805  7937.549805  7781.100098  7924.549805  134700
2016-01-05  7784.649902  7831.200195  7763.250000  7828.399902  145200
2016-01-06  7741.000000  7800.950195  7721.200195  7788.049805  147100
2016-01-07  7568.299805  7674.950195  7556.600098  7673.350098  188900
2016-01-08  7601.350098  7634.100098  7581.049805  7611.649902  157400


In [7]:
nifty = yf.download("^NSEI", start="2024-01-01", end="2024-12-31", interval="1d")
print(nifty.head())
nifty.to_csv("nifty50_data2024.csv")

C:\Users\RITANKAR\AppData\Local\Temp\ipykernel_4736\1206801362.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI", start="2024-01-01", end="2024-12-31", interval="1d")
[*********************100%***********************]  1 of 1 completed

Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2024-01-01  21741.900391  21834.349609  21680.849609  21727.750000  154000
2024-01-02  21665.800781  21755.599609  21555.650391  21751.349609  263700
2024-01-03  21517.349609  21677.000000  21500.349609  21661.099609  311900
2024-01-04  21658.599609  21685.650391  21564.550781  21605.800781  339200
2024-01-05  21710.800781  21749.599609  21629.199219  21705.750000  309300


In [3]:
import pandas as pd
import numpy as np 
import ta
from sklearn.preprocessing import MinMaxScaler

In [18]:
df_2016 = pd.read_csv('nifty50_data2016.csv', parse_dates=['Date'])
df_2016.sort_values('Date', inplace=True)

df_2024 = pd.read_csv('nifty50_data2024.csv', parse_dates = ['Date'])
df_2024.sort_values('Date', inplace = True)


In [19]:
## NOW WE ADD THE TAs
def add_technical_indicators(df):
    df['SMA_10'] = ta.trend.SMAIndicator(df['Close'], window=10).sma_indicator()
    df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
    df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
    
    macd = ta.trend.MACD(df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    
    bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
    df['BB_high'] = bb.bollinger_hband()
    df['BB_low'] = bb.bollinger_lband()
    df['BB_width'] = df['BB_high'] - df['BB_low']
    
    df['Momentum'] = ta.momentum.WilliamsRIndicator(
        high=df['High'], low=df['Low'], close=df['Close'], lbp=14
    ).williams_r()
    
    return df

In [20]:
df_2016 = add_technical_indicators(df_2016)
df_2024 = add_technical_indicators(df_2024)

In [21]:
### TARGET FOR UP/DOWN
def create_target(df):
    df['Next_Close'] = df['Close'].shift(-1)
    df['UpDown'] = np.where(df['Next_Close'] > df['Close'], 1, 0)
    return df.dropna(subset=['UpDown'])

In [22]:
df_2016 = create_target(df_2016)
df_2024 = create_target(df_2024)

In [26]:
df_2024.head()


,Date,Close,High,Low,Open,Volume,SMA_10,SMA_20,RSI,MACD,MACD_signal,BB_high,BB_low,BB_width,Momentum,Next_Close,UpDown
0,01-01-2024,21741.90039,21834.34961,21680.84961,21727.75000,154000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21697.44922,0
21,01-02-2024,21697.44922,21832.94922,21658.75000,21780.65039,332500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22338.75000,1
42,01-03-2024,22338.75000,22353.30078,22047.75000,22048.30078,351500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22462.00000,1
60,01-04-2024,22462.00000,22529.94922,22427.75000,22455.00000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24141.94922,1
120,01-07-2024,24141.94922,24164.00000,23992.69922,23992.94922,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25010.90039,1


In [27]:
df_2016.head()

,Date,Close,High,Low,Open,Volume,SMA_10,SMA_20,RSI,MACD,MACD_signal,BB_high,BB_low,BB_width,Momentum,Next_Close,UpDown
0,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7784.649902,0
1,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7741.000000,0
2,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7568.299805,0
3,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7601.350098,1
4,2016-01-08,7601.350098,7634.100098,7581.049805,7611.649902,157400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7563.850098,0


In [29]:
df_2016.dropna(inplace=True)
df_2024.dropna(inplace=True)

In [31]:
features = ['Open', 'High', 'Low', 'Close', 'Volume',
            'SMA_10', 'SMA_20', 'RSI', 'MACD', 'MACD_signal',
            'BB_high', 'BB_low', 'BB_width', 'Momentum']

scaler = MinMaxScaler()
df_2016[features] = scaler.fit_transform(df_2016[features])
df_2024[features] = scaler.transform(df_2024[features])

In [32]:
df_2016.to_csv('processed_2016.csv', index=False)
df_2024.to_csv('processed_2024.csv', index=False)



In [42]:
df_pro2016 = pd.read_csv('processed_2016.csv')


In [ ]:
# Since, in a check while running the model, we found the accuracy to be 0.53, which is very low, so here are some checks and cleaning done before feeding the model

In [45]:
df_pro2016.isnull().sum()  #value null or not

Date           0
Close          0
High           0
Low            0
Open           0
Volume         0
SMA_10         0
SMA_20         0
RSI            0
MACD           0
MACD_signal    0
BB_high        0
BB_low         0
BB_width       0
Momentum       0
Next_Close     0
UpDown         0
dtype: int64

In [50]:
df_pro2016.duplicated(subset='Date').sum()

np.int64(0)

In [51]:
df_pro2016['UpDown'].value_counts(normalize=True)

UpDown
1    0.542857
0    0.457143
Name: proportion, dtype: float64

In [ ]:
##MODEL TRAINING ON 2016

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df_data = pd.read_csv('processed_2016.csv')

X_2016 = df_2016.drop(columns=['UpDown', 'Date'])
y_2016 = df_2016['UpDown']

X_train, X_test, y_train, y_test = train_test_split(X_2016, y_2016, test_size=0.2, shuffle=False)



In [78]:
model = RandomForestClassifier(n_estimators=210, random_state=42)
model.fit(X_train, y_train)

,n_estimators,210
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [79]:
y_pred = model.predict(X_test)

In [80]:
print("Accuracy on 2016 Test Data:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy on 2016 Test Data: 0.5238095238095238
              precision    recall  f1-score   support

           0       0.75      0.14      0.23        22
           1       0.50      0.95      0.66        20

    accuracy                           0.52        42
   macro avg       0.62      0.54      0.44        42
weighted avg       0.63      0.52      0.43        42



In [ ]:
### NOW Evaluate on 2024

In [64]:
df_2024 = pd.read_csv('processed_2024.csv')
X_2024 = df_2024.drop(columns=['UpDown', 'Date'])
y_2024 = df_2024['UpDown']



In [65]:
y_pred_2024 = model.predict(X_2024)

In [66]:
print("Accuracy on 2024 Data:", accuracy_score(y_2024, y_pred_2024))
print(classification_report(y_2024, y_pred_2024))

Accuracy on 2024 Data: 0.4075829383886256
              precision    recall  f1-score   support

           0       0.40      1.00      0.58        85
           1       1.00      0.01      0.02       126

    accuracy                           0.41       211
   macro avg       0.70      0.50      0.30       211
weighted avg       0.76      0.41      0.24       211

